In [31]:
import time
import requests
import pandas as pd

In [33]:
HEADERS = {"User-Agent": "BookDatasetBuilder/1.0 (contact: you@example.com)"}

# Mapea la lista de géneros -> el "subject" que usa Open Library en la URL.

GENRE_TO_SUBJECT = {
    "Fantasy": "fantasy",
    "Science fiction": "science_fiction",
    "Mystery": "mystery",
    "Romance": "romance",
    "Horror": "horror",
    "Historical fiction": "historical_fiction",
    "Young adult": "young_adult",
    "Non fiction": "nonfiction",
}

def fetch_subject_works(subject_slug: str, limit: int = 200, sleep_s: float = 0.2) -> list[dict]:
    """
    Open Library Subjects API:
    GET /subjects/{subject}.json?limit=...
    Devuelve obras asociadas al subject.  :contentReference[oaicite:1]{index=1}
    """
    url = f"https://openlibrary.org/subjects/{subject_slug}.json"
    params = {"limit": limit, "details": "true"}  # details añade info útil cuando esté disponible
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    time.sleep(sleep_s)
    return data.get("works", [])

def best_year(work: dict):
    # subjects API suele traer first_publish_year o first_publish_date
    y = work.get("first_publish_year")
    if isinstance(y, int):
        return y
    d = work.get("first_publish_date")
    if isinstance(d, str) and len(d) >= 4 and d[:4].isdigit():
        return int(d[:4])
    return None

def best_author(work: dict):
    authors = work.get("authors") or []
    if authors and isinstance(authors, list):
        name = authors[0].get("name")
        return name
    return None

# def best_publisher(work: dict):
    # Subjects API no siempre trae editorial. A veces aparece en 'publishers' dentro de details.
    # Si no está, queda None.
#    pubs = work.get("publishers") or work.get("publisher") or []
#    if isinstance(pubs, list) and pubs:
#        return pubs[0]
#    if isinstance(pubs, str):
#        return pubs
#    return None

def build_dataset(per_genre: int = 200) -> pd.DataFrame:
    rows = []
    for genre, subject in GENRE_TO_SUBJECT.items():
        works = fetch_subject_works(subject, limit=per_genre)

        for w in works[:per_genre]:
            rows.append({
                "title": w.get("title"),
                "author": best_author(w),
                "year": best_year(w),
                # "publisher": best_publisher(w),
                "genre": genre,
                "openlibrary_work_key": w.get("key"),
            })

    df = pd.DataFrame(rows)
    return df

def basic_clean(df) -> pd.DataFrame:

    # Limpieza básica
    for c in ["title", "author", "genre"]:
        df[c] = df[c].astype("string").str.strip()

    df = df[df["title"].notna() & (df["title"].str.len() > 0)].copy()

    # Deduplicación simple (título + autor + año)
    df["dedupe"] = (
        df["title"].fillna("") + "||" +
        df["author"].fillna("") + "||" +
        df["year"].fillna(-1).astype("int64").astype("string")
    )
    df = df.drop_duplicates("dedupe").drop(columns=["dedupe"])

    return df

if __name__ == "__main__":
    df = build_dataset(per_genre=200)  # ajustar a 50/200/1000 según se quiera. ¡Es por género!
    df.to_csv("books_multi_genre.csv", index=False, encoding="utf-8")
    # df = basic_clean(df)
    print("Filas:", len(df))
    print(df.head(10).to_string(index=False))

Filas: 1600
                           title                 author  year   genre openlibrary_work_key
Alice's Adventures in Wonderland          Lewis Carroll  1865 Fantasy     /works/OL138052W
      The Wonderful Wizard of Oz          L. Frank Baum  1899 Fantasy      /works/OL18417W
                 Treasure Island Robert Louis Stevenson  1880 Fantasy      /works/OL24034W
              Gulliver's Travels         Jonathan Swift  1726 Fantasy      /works/OL20600W
       A Midsummer Night's Dream    William Shakespeare  1600 Fantasy     /works/OL259010W
                      The Prince    Niccolò Machiavelli  1515 Fantasy    /works/OL1089297W
       Through the Looking-Glass          Lewis Carroll  1865 Fantasy     /works/OL151406W
         The Wind in the Willows        Kenneth Grahame  1908 Fantasy   /works/OL28570037W
            Five Children and It           Edith Nesbit  1905 Fantasy      /works/OL99499W
     The Princess and the Goblin       George MacDonald  1872 Fantasy      /wo

In [3]:
df['genre'].value_counts(dropna=False)

genre
Fantasía           200
Ciencia ficción    200
Misterio           200
Romance            200
Terror             200
Histórica          200
Juvenil            200
No ficción         200
Name: count, dtype: int64

In [4]:
df = basic_clean(df)
print("Filas:", len(df))
df.head(10)

Filas: 1461


,title,author,year,publisher,genre,openlibrary_work_key
0,Alice's Adventures in Wonderland,Lewis Carroll,1865,<NA>,Fantasía,/works/OL138052W
1,The Wonderful Wizard of Oz,L. Frank Baum,1899,<NA>,Fantasía,/works/OL18417W
2,Treasure Island,Robert Louis Stevenson,1880,<NA>,Fantasía,/works/OL24034W
3,Gulliver's Travels,Jonathan Swift,1726,<NA>,Fantasía,/works/OL20600W
4,A Midsummer Night's Dream,William Shakespeare,1600,<NA>,Fantasía,/works/OL259010W
5,The Prince,Niccolò Machiavelli,1515,<NA>,Fantasía,/works/OL1089297W
6,Through the Looking-Glass,Lewis Carroll,1865,<NA>,Fantasía,/works/OL151406W
7,The Wind in the Willows,Kenneth Grahame,1908,<NA>,Fantasía,/works/OL28570037W
8,Five Children and It,Edith Nesbit,1905,<NA>,Fantasía,/works/OL99499W
9,The Princess and the Goblin,George MacDonald,1872,<NA>,Fantasía,/works/OL15449W


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1461 entries, 0 to 1599
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   title                 1461 non-null   string
 1   author                1461 non-null   string
 2   year                  1461 non-null   int64 
 3   publisher             0 non-null      string
 4   genre                 1461 non-null   string
 5   openlibrary_work_key  1461 non-null   object
dtypes: int64(1), object(1), string(4)
memory usage: 79.9+ KB


In [6]:
df['genre'].value_counts(dropna=False)

genre
Fantasía           200
Misterio           195
No ficción         194
Romance            188
Juvenil            180
Terror             177
Histórica          164
Ciencia ficción    163
Name: count, dtype: Int64

In [7]:
# borrar la columna 'publisher' o directamente la función o falsear la columna

# Editoriales:
#################
# Penguin Random House
# Hachette UK
# Bloomsbury
# Harper Collins
# Virago Press
# Routledge
# Persephone Books
# Hogarth Press
# Michael Joseph
# Orbit Books

In [8]:
df['year'].min()

0

In [9]:
print(f"El intervalo de años de publicación va de {df['year'].min()} a {df['year'].max()}")

El intervalo de años de publicación va de 0 a 2025


In [10]:
for row in df.itertuples(): 
    if row.year == 0:
        print(f'El título: {row.title} tiene un cero ({row.year}) como año de publicación.')
    

El título: The Invisible Man tiene un cero (0) como año de publicación.
El título: The War of the Worlds tiene un cero (0) como año de publicación.


The Invisible Man -> 1897
The War of the Worlds -> 1898

In [ ]:
# Eliminar los libros con año = 0
# df = df[df['year'] != 0]
# print(f"El intervalo de años de publicación va de {df['year'].min()} a {df['year'].max()}")

In [ ]:
# df2 = pd.read_csv('books_multi_genre.csv')
# df2.head(10)

,title,author,year,publisher,genre,openlibrary_work_key
0,Alice's Adventures in Wonderland,Lewis Carroll,1865,NaN,Fantasía,/works/OL138052W
1,The Wonderful Wizard of Oz,L. Frank Baum,1899,NaN,Fantasía,/works/OL18417W
2,Treasure Island,Robert Louis Stevenson,1880,NaN,Fantasía,/works/OL24034W
3,Gulliver's Travels,Jonathan Swift,1726,NaN,Fantasía,/works/OL20600W
4,A Midsummer Night's Dream,William Shakespeare,1600,NaN,Fantasía,/works/OL259010W
5,The Prince,Niccolò Machiavelli,1515,NaN,Fantasía,/works/OL1089297W
6,Through the Looking-Glass,Lewis Carroll,1865,NaN,Fantasía,/works/OL151406W
7,The Wind in the Willows,Kenneth Grahame,1908,NaN,Fantasía,/works/OL28570037W
8,Five Children and It,Edith Nesbit,1905,NaN,Fantasía,/works/OL99499W
9,The Princess and the Goblin,George MacDonald,1872,NaN,Fantasía,/works/OL15449W


## Por hacer:
- [ ] publishers eliminar
- [x] Cambiar géneros a inglés
- [ ] Añadir años de publicación para los dos libros que están en año 0
- [ ] Sacar sinopsis 
- [ ] décadas: 
1400-1499, 
1500-1599, 
1600-1699, 
1700-1799, 
1800-1849, 
1850-1899, 
1900-1949, 
1950-1969, 
1970-1989, 
1990-1999, 
2000-2009, 
2010-2019, 
2020-2026

- [ ] IRENE: csv titulo y sinopsis

- [ ] CLARA: csv sin duplicados pero con los años ajustados y demás, csv con duplicados